# BDA Blueprint Instruction Optimization - Purchase Order Workshop

This notebook demonstrates how to use the Amazon Bedrock Data Automation (BDA) Blueprint Instruction Optimization API to automatically improve extraction accuracy for purchase order documents.

**What this notebook does:**
1. Initialize clients and S3 bucket
2. Upload a sample purchase order document to S3
3. Create a BDA blueprint from a JSON schema
4. Review the ground truth data used to measure accuracy
5. Run optimization using 3 labeled document samples
6. Poll until the async optimization job completes
7. Retrieve the optimized blueprint and compare updated instructions
8. (Optional) Promote the optimized blueprint to LIVE for production use
9. Clean up resources

**How optimization works:** BDA compares its extraction results against your ground truth values, then rewrites the natural language instructions for each field to improve accuracy. You provide the documents and correct answers — BDA handles the rest.

**Prerequisites:** AWS account with BDA access, configured credentials, Python 3.8+

In [ ]:
import boto3
import json
import time
import os
from IPython.display import display, IFrame

print(f"boto3 version: {boto3.__version__}")

## 1. Initialize clients and S3 bucket

Set up the BDA and S3 clients, and resolve the data automation profile ARN.

The **data automation profile** tells BDA which inference profile to use for processing. The profile ARN follows a fixed pattern based on your region and account. The `BDA_BUCKET` environment variable is set by the CloudFormation stack — it's the S3 bucket where input documents and optimization output are stored.

In [ ]:
session = boto3.Session()
region = session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']

bda_client = boto3.client('bedrock-data-automation', region_name=region)
s3_client = boto3.client('s3', region_name=region)

# S3 bucket provisioned by the CloudFormation stack
bucket_name = os.environ['BDA_BUCKET']

# The data automation profile ARN is region-scoped and uses cross-region inference
profile_arn = f"arn:aws:bedrock:{region}:{account_id}:data-automation-profile/us.data-automation-v1"

print(f"Region: {region}")
print(f"Account: {account_id}")
print(f"Bucket: {bucket_name}")
print(f"Profile ARN: {profile_arn}")

## 2. Upload sample purchase order to S3

BDA reads input documents directly from S3, so we upload the sample PDF before running any extraction or optimization. This document is from Acme Bikes' supplier Mountain Gear Co. and represents one of the purchase order variants in the dataset.

We also render the PDF inline so you can see what BDA will be extracting from.

In [ ]:
local_pdf_path = "sample_data/purchase_order/PO_RT-005_Speed_Shop.pdf"

s3_key = "bda/input/purchase_order/PO_RT-005_Speed_Shop.pdf"
s3_client.upload_file(local_pdf_path, bucket_name, s3_key)
document_s3_uri = f"s3://{bucket_name}/{s3_key}"
print(f"Uploaded to: {document_s3_uri}")

In [ ]:
# Render the PDF so you can see the document layout and field locations
display(IFrame(local_pdf_path, width=800, height=500))

## 3. Create blueprint with `CreateBlueprint`

A **blueprint** defines which fields to extract from a document and how. Each field has a name and a natural language `instruction` that guides BDA's extraction model.

We load the schema from `bda_schema.json`, which defines fields like `po_number`, `vendor_name`, `line_items`, etc. We also snapshot the original instructions now so we can compare them against the optimized versions in step 7.

The blueprint is created in `DEVELOPMENT` stage, which is a safe sandbox — changes here don't affect any `LIVE` production blueprint until you explicitly promote them.

In [ ]:
with open('sample_data/purchase_order/bda_schema.json') as f:
    blueprint_schema = json.load(f)

# Snapshot original instructions and definitions for before/after comparison in step 7
import copy
original_instructions = {
    name: prop["instruction"]
    for name, prop in blueprint_schema["properties"].items()
}
original_definitions = copy.deepcopy(blueprint_schema.get("definitions", {}))

print("Blueprint schema:")
print(json.dumps(blueprint_schema, indent=2))

In [ ]:
# Append a timestamp to the name to avoid conflicts if you run this notebook multiple times
blueprint_name = f"acme-bikes-purchase-order-{int(time.time())}"

response = bda_client.create_blueprint(
    blueprintName=blueprint_name,
    type='DOCUMENT',
    blueprintStage='DEVELOPMENT',  # Safe sandbox — won't affect LIVE until promoted
    schema=json.dumps(blueprint_schema)
)

blueprint_arn = response['blueprint']['blueprintArn']
print(f"Blueprint created: {blueprint_name}")
print(f"ARN: {blueprint_arn}")

## 4. Review ground truth data

**Ground truth** is the set of correct expected values for each field in a document. During optimization, BDA compares its extraction output against these values to measure accuracy and identify which instructions need improvement.

Ground truth quality directly affects optimization quality — incorrect or incomplete ground truth will produce misleading results. Review the values below and verify they match what you see in the PDF rendered above.

In [ ]:
gt_path = "sample_data/purchase_order/PO_RT-005_Speed_Shop_ground_truth.json"

with open(gt_path) as f:
    ground_truth = json.load(f)

print("Ground truth for Speed Shop PO:")
print(json.dumps(ground_truth, indent=2))

## 5. Run blueprint optimization with `InvokeBlueprintOptimizationAsync`

Optimization requires at least 3 document samples, each paired with a ground truth file. Using more samples (up to 10) and including edge cases — documents where extraction has previously failed — produces stronger results.

Here we use all 6 available purchase orders from different Acme Bikes suppliers: Speed Shop, Trail Riders, City Bikes Plus, Wheels & More, Bike Depot, and Bike Emporium. Using more samples gives BDA greater variety to generalize the optimized instructions.

The API is **asynchronous** — it returns an `invocationArn` immediately and runs the job in the background. We poll for completion in the next step.

In [ ]:
# Each entry pairs a PDF document with its corresponding ground truth JSON
samples_config = [
    {
        'pdf': 'sample_data/purchase_order/PO_RT-005_Speed_Shop.pdf',
        'gt':  'sample_data/purchase_order/PO_RT-005_Speed_Shop_ground_truth.json',
        's3_prefix': 'bda/input/purchase_order/PO_RT-005_Speed_Shop'
    },
    {
        'pdf': 'sample_data/purchase_order/PO_RT-006_Trail_Riders.pdf',
        'gt':  'sample_data/purchase_order/PO_RT-006_Trail_Riders_ground_truth.json',
        's3_prefix': 'bda/input/purchase_order/PO_RT-006_Trail_Riders'
    },
    {
        'pdf': 'sample_data/purchase_order/PO_RT-007_City_Bikes_Plus.pdf',
        'gt':  'sample_data/purchase_order/PO_RT-007_City_Bikes_Plus_ground_truth.json',
        's3_prefix': 'bda/input/purchase_order/PO_RT-007_City_Bikes_Plus'
    },
    {
        'pdf': 'sample_data/purchase_order/PO_RT-011_Wheels__More.pdf',
        'gt':  'sample_data/purchase_order/PO_RT-011_Wheels__More_ground_truth.json',
        's3_prefix': 'bda/input/purchase_order/PO_RT-011_Wheels__More'
    },
    {
        'pdf': 'sample_data/purchase_order/PO_RT-013_Bike_Depot.pdf',
        'gt':  'sample_data/purchase_order/PO_RT-013_Bike_Depot_ground_truth.json',
        's3_prefix': 'bda/input/purchase_order/PO_RT-013_Bike_Depot'
    },
    {
        'pdf': 'sample_data/purchase_order/PO_RT-015_Bike_Emporium.pdf',
        'gt':  'sample_data/purchase_order/PO_RT-015_Bike_Emporium_ground_truth.json',
        's3_prefix': 'bda/input/purchase_order/PO_RT-015_Bike_Emporium'
    }
]


# Upload each PDF and ground truth file to S3, then build the samples list for the API call
samples = []
for s in samples_config:
    pdf_key = f"{s['s3_prefix']}.pdf"
    gt_key  = f"{s['s3_prefix']}_ground_truth.json"
    s3_client.upload_file(s['pdf'], bucket_name, pdf_key)
    s3_client.upload_file(s['gt'],  bucket_name, gt_key)
    samples.append({
        'assetS3Object':       {'s3Uri': f"s3://{bucket_name}/{pdf_key}"},
        'groundTruthS3Object': {'s3Uri': f"s3://{bucket_name}/{gt_key}"}
    })
    print(f"Uploaded: {s['s3_prefix']}")

print(f"\n{len(samples)} sample sets ready")

In [ ]:
# BDA writes optimization results (metrics + updated schema) to this S3 prefix
optimization_output_uri = f"s3://{bucket_name}/bda/optimization-output/purchase-order/"

response = bda_client.invoke_blueprint_optimization_async(
    blueprint={
        'blueprintArn': blueprint_arn,
        'stage': 'DEVELOPMENT'
    },
    samples=samples,
    outputConfiguration={
        's3Object': {'s3Uri': optimization_output_uri}
    },
    dataAutomationProfileArn=profile_arn
)

# Save the invocation ARN — we need it to poll status and retrieve results
invocation_arn = response['invocationArn']
print(f"Optimization job started")
print(f"Invocation ARN: {invocation_arn}")

## 6. Poll status with `GetBlueprintOptimizationStatus`

Optimization typically completes in a few minutes depending on document count and page length. This cell polls every 15 seconds and exits when the job reaches a terminal state (`Success`, `ServiceError`, or `ClientError`).

Once the job succeeds, BDA has updated the blueprint's field instructions in-place. The next step retrieves the updated blueprint.

In [ ]:
print("Polling optimization status...\n")

while True:
    status_response = bda_client.get_blueprint_optimization_status(
        invocationArn=invocation_arn
    )
    status = status_response.get('status', 'Unknown')
    print(f"  Status: {status}")

    if status == 'Success':
        print(f"\nOptimization completed.")
        break
    elif status in ('ServiceError', 'ClientError'):
        print(f"\nOptimization failed: {status_response.get('errorMessage', 'N/A')}")
        break

    time.sleep(15)

## 7. Retrieve optimized blueprint with `GetBlueprint`

Optimization updates the blueprint's field instructions directly in the `DEVELOPMENT` stage. We fetch the updated schema and compare each field's instruction against the original to see what changed.

Fields marked `[CHANGED]` have new instructions that incorporate patterns BDA learned from your samples — for example, adding layout hints like "typically found in the upper right corner" or clarifying ambiguous label names. Fields marked `[UNCHANGED]` were already extracting correctly and didn't need refinement.

In [ ]:
bp_response = bda_client.get_blueprint(
    blueprintArn=blueprint_arn,
    blueprintStage='DEVELOPMENT'
)

optimized_schema = json.loads(bp_response['blueprint']['schema'])

print("Optimized schema:")
print(json.dumps(optimized_schema, indent=2))

In [ ]:
def compare_instructions(original_props, optimized_props, orig_defs=None, opt_defs=None, indent=0):
    """Recursively compare field instructions between original and optimized schemas.
    Handles nested objects, arrays, and $ref definitions."""
    prefix = '  ' * indent
    orig_defs = orig_defs or {}
    opt_defs  = opt_defs  or {}
    for field_name, orig_prop in original_props.items():
        if '$ref' in orig_prop:
            orig_prop = orig_defs.get(orig_prop['$ref'].split('/')[-1], orig_prop)
        opt_prop = optimized_props.get(field_name, {})
        if '$ref' in opt_prop:
            opt_prop = opt_defs.get(opt_prop['$ref'].split('/')[-1], opt_prop)

        if 'instruction' in orig_prop:
            orig_instr = orig_prop['instruction']
            opt_instr  = opt_prop.get('instruction', 'N/A')
            changed = orig_instr != opt_instr
            print(f"\n{prefix}{field_name} {'[CHANGED]' if changed else '[UNCHANGED]'}")
            print(f"{prefix}  Before: {orig_instr}")
            print(f"{prefix}  After:  {opt_instr}")

        if orig_prop.get('type') == 'array' and 'items' in orig_prop:
            orig_items = orig_prop['items']
            if '$ref' in orig_items:
                orig_items = orig_defs.get(orig_items['$ref'].split('/')[-1], {})
            opt_items = opt_prop.get('items', {})
            if '$ref' in opt_items:
                opt_items = opt_defs.get(opt_items['$ref'].split('/')[-1], {})
            if 'properties' in orig_items:
                print(f"{prefix}  [{field_name} items]")
                compare_instructions(orig_items['properties'], opt_items.get('properties', {}), orig_defs, opt_defs, indent + 2)

        elif orig_prop.get('type') == 'object' and 'properties' in orig_prop:
            compare_instructions(orig_prop['properties'], opt_prop.get('properties', {}), orig_defs, opt_defs, indent + 1)

print("Instruction comparison (before -> after optimization):")
print("=" * 80)

compare_instructions(
    blueprint_schema['properties'],
    optimized_schema.get('properties', {}),
    orig_defs=original_definitions,
    opt_defs=optimized_schema.get('definitions', {})
)

print("\n" + "=" * 80)

## 8. (Optional) Promote to LIVE

Blueprints have two stages: `DEVELOPMENT` (sandbox) and `LIVE` (production). Optimization always runs against `DEVELOPMENT`. Once you're satisfied with the results, promote the blueprint to `LIVE` to make the optimized instructions active for production workloads.

Uncomment the code below to promote.

In [ ]:
# Uncomment to copy the optimized DEVELOPMENT blueprint to LIVE:
# bda_client.copy_blueprint_stage(
#     blueprintArn=blueprint_arn,
#     sourceStage='DEVELOPMENT',
#     targetStage='LIVE'
# )
# print(f"Blueprint promoted to LIVE: {blueprint_arn}")

print("Uncomment the code above to promote the optimized blueprint to LIVE stage.")

## 9. Clean up

Run this cell to delete the blueprint and empty the S3 bucket. The bucket must be empty before CloudFormation can delete it — so run this before deleting the stack.

In [ ]:
# Delete the blueprint
bda_client.delete_blueprint(blueprintArn=blueprint_arn)
print(f"Deleted blueprint: {blueprint_arn}")

# Empty the S3 bucket (required before CloudFormation can delete it)
paginator = s3_client.get_paginator('list_object_versions')
for page in paginator.paginate(Bucket=bucket_name):
    objects = [
        {'Key': o['Key'], 'VersionId': o['VersionId']}
        for o in page.get('Versions', []) + page.get('DeleteMarkers', [])
    ]
    if objects:
        s3_client.delete_objects(Bucket=bucket_name, Delete={'Objects': objects, 'Quiet': True})

print(f"Emptied S3 bucket: {bucket_name}")
print("You can now delete the CloudFormation stack.")